# Label preprocessing

**In plain language:** We clip the CT gray values so bleeding stands out, then feed the same picture to a fast 2D model (MONAI) and a slower 3D model (nnU-Net).

Research software — not for clinical use. Uses `data/demo/` only.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import yaml

cwd = Path.cwd().resolve()
if (cwd / "config" / "preprocessing.yaml").is_file():
    ROOT = cwd
elif (cwd.parent / "config" / "preprocessing.yaml").is_file():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Run this notebook from the repo root or notebooks/")

src = ROOT / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

CT = ROOT / "data" / "demo" / "01_all_classes" / "all_classes.nii.gz"
GT = ROOT / "data" / "demo" / "ground_truth" / "01_all_classes" / "all_classes.nii.gz"
cfg = yaml.safe_load((ROOT / "config" / "preprocessing.yaml").read_text(encoding="utf-8"))
CLIP_MIN, CLIP_MAX = cfg["clip_min"], cfg["clip_max"]

volume = nib.load(CT).get_fdata().astype(np.float32)
mask = nib.load(GT).get_fdata().astype(np.int16)
windowed = np.clip(volume, CLIP_MIN, CLIP_MAX)
z = int(np.argmax((mask > 0).sum(axis=(0, 1))))
print(f"project: {ROOT}")
print(f"shape: {volume.shape}  HU window: [{CLIP_MIN}, {CLIP_MAX}]  slice: {z}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(np.rot90(volume[:, :, z]), cmap="gray")
axes[0].set_title("Raw CT")
axes[0].axis("off")
axes[1].imshow(np.rot90(windowed[:, :, z]), cmap="gray")
axes[1].set_title(f"Windowed [{CLIP_MIN}, {CLIP_MAX}] HU")
axes[1].axis("off")
axes[2].imshow(np.rot90(mask[:, :, z]), cmap="nipy_spectral", vmin=0, vmax=5)
axes[2].set_title("Expert labels")
axes[2].axis("off")
plt.tight_layout()
plt.show()
print("MONAI sees three neighboring 2D slices. nnU-Net sees the full 3D volume. Same window for both.")

**What you are looking at:** the middle picture is the same scan after we throw away gray values that hide blood. MONAI is the fast demo (Dice 0.257). nnU-Net is the stronger 3D model (Dice 0.455).